# Phase 2 Architecture A v2

Shared-backbone, multi-head XGBoost baseline with the v1 verdict fixes applied.

This notebook is intentionally unexecuted. Upload it to Google Colab, upload the merged CSV, then run cells top to bottom.

## v2 Changes

- 5-fold stratified cross-validation instead of a single 80/20 split.
- Balanced sample weights for every head.
- PCA increased from 35 to 40 components.
- Expanded hand-crafted features from 21 to 42.
- Two-stage training: first train tier/categorical heads, then append predicted tier for `d1`-`d5` heads.
- `formatting` task type is merged into `generation` because it has only 2 samples.
- Defensive fix for any remaining Phase 1 question prompts mislabeled as `classification`.
- Confidence uses the mean of key categorical-head probabilities instead of the minimum across all heads.

In [6]:
# Colab setup
!pip -q install sentence-transformers xgboost

In [7]:
import ast
import json
import re
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

from sentence_transformers import SentenceTransformer
from sklearn.decomposition import PCA
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.utils.class_weight import compute_sample_weight
from xgboost import XGBClassifier

warnings.filterwarnings('ignore')

RANDOM_STATE = 42
PCA_COMPONENTS = 40
N_SPLITS = 5

## Load Dataset

Upload `prompt_classifier_phase1_phase2_merged_cleaned.csv` to Colab, then update `DATA_PATH` if needed.

In [9]:
DATA_PATH = '/content/prompt_classifier_phase1_phase2_merged_cleaned.csv'

df = pd.read_csv(DATA_PATH)
print('Loaded:', df.shape)
df.head()

Loaded: (2273, 24)


,id,prompt,intent,task_type,reasoning_chain_detected,d1,d2,d3,d4,d5,...,research_signals,confidence,low_confidence_flag,task_description,expected_answer,prompting_techniques,prompt_type,phrasing_style,domain,source
0,NaN,Imagine you are a science fiction author renow...,SYNTHETIC,generation,True,0.75,0.50,0.50,0.0,0.0,...,[],0.80,False,Develop a creative narrative about neural pros...,The ideal output would be a well-structured an...,"['ROLE_PROMPTING', 'TREE_OF_THOUGHTS']",CREATIVE_WRITING,NaN,NaN,phase2
1,NaN,You are an expert photography tutor. I want to...,ANALYTICAL,generation,True,0.50,0.75,0.50,0.0,0.0,...,[],0.80,False,Decode the logic behind this photography techn...,The ideal output would be a Python function th...,['CODE_PROMPTING'],CODE_EXPLANATION,NaN,NaN,phase2
2,NaN,You are a leading neuroscientist specializing ...,ANALYTICAL,reasoning,True,0.75,0.75,0.75,0.5,0.5,...,"[""scientific""]",0.90,False,Chat about recent developments in neural prost...,The ideal answer would begin with a brief defi...,"['ROLE_PROMPTING', 'CHAIN_OF_THOUGHT']",CONVERSATIONAL,NaN,NaN,phase2
3,NaN,I want to understand the basic human emotions....,FACTUAL,generation,False,0.00,0.50,0.50,0.0,0.0,...,[],0.95,False,Distinguish between various basic emotions tec...,"An ideal answer would first define emotion, th...","['CHAIN_OF_THOUGHT', 'CONTEXTUAL_PROMPTING']",COMPARISON,NaN,NaN,phase2
4,NaN,Here are examples of competitive exclusion. Ex...,ANALYTICAL,reasoning,True,0.75,0.50,0.50,0.0,0.5,...,[],0.80,False,Code a solution for theoretical ecology,The principle of competitive exclusion states ...,['ONE_SHOT_FEW_SHOT'],PROGRAMMING_CODE_GENERATION,NaN,NaN,phase2


## Dataset Safety Fixes and Validation

The cleaned shared dataset should already contain the fixes, but this cell is defensive so the notebook remains safe if someone runs it against an older copy.

In [10]:
SCORE_COLS = ['d1', 'd2', 'd3', 'd4', 'd5']
VALID_SCORES = [0.0, 0.25, 0.5, 0.75, 1.0]
SCORE_TO_CLASS = {score: idx for idx, score in enumerate(VALID_SCORES)}
CLASS_TO_SCORE = {idx: score for score, idx in SCORE_TO_CLASS.items()}

DIMENSION_LABELS = {
    'd1': 'Semantic Complexity',
    'd2': 'Domain Specificity',
    'd3': 'Output Formality',
    'd4': 'Research Dependency',
    'd5': 'Context Requirement',
}


def normalize_bool(value):
    if isinstance(value, bool):
        return value
    if pd.isna(value):
        return False
    return str(value).strip().lower() == 'true'


def has_any_term(text, terms):
    for term in terms:
        if re.search(rf'(?<![A-Za-z0-9_]){re.escape(term)}(?![A-Za-z0-9_])', text):
            return True
    return False


def rederive_question_task_type(prompt):
    lower = str(prompt).lower()
    if has_any_term(lower, ['python', 'sql query', 'source code', 'code', 'function', 'script', 'debug', 'yaml', 'json', 'syntax error', 'stack trace', 'kubernetes manifest']):
        return 'coding'
    if has_any_term(lower, ['summarize', 'summary', 'tl;dr', 'condense']):
        return 'summarisation'
    if has_any_term(lower, ['create', 'draft', 'write', 'generate', 'compose', 'build']):
        return 'generation'
    return 'reasoning'


def complexity_score_from_dims_frame(frame):
    return (
        frame['d1'] * 0.35 +
        frame['d2'] * 0.20 +
        frame['d3'] * 0.20 +
        frame['d4'] * 0.15 +
        frame['d5'] * 0.10
    )


def tier_from_score(score):
    if score < 0.40:
        return 'T1'
    if score < 0.70:
        return 'T2'
    return 'T3'


def parse_research_signals(value):
    if pd.isna(value):
        return []
    if isinstance(value, list):
        return value
    try:
        parsed = json.loads(value)
    except Exception:
        try:
            parsed = ast.literal_eval(str(value))
        except Exception:
            return []
    return parsed if isinstance(parsed, list) else []

# Defensive label cleanup from the v1 verdict.
question_classification_mask = (
    (df.get('source', '') == 'phase1') &
    df['prompt'].astype(str).str.strip().str.endswith('?') &
    (df['task_type'] == 'classification')
)
fixed_question_count = int(question_classification_mask.sum())
if fixed_question_count:
    df.loc[question_classification_mask, 'task_type'] = df.loc[question_classification_mask, 'prompt'].apply(rederive_question_task_type)

formatting_count = int((df['task_type'] == 'formatting').sum())
df.loc[df['task_type'] == 'formatting', 'task_type'] = 'generation'

df['reasoning_chain_detected'] = df['reasoning_chain_detected'].apply(normalize_bool)
df['research_signals_parsed'] = df['research_signals'].apply(parse_research_signals)
df['computed_complexity_score'] = complexity_score_from_dims_frame(df)
df['computed_tier'] = df['computed_complexity_score'].apply(tier_from_score)

print(f'Question task_type fixes applied: {fixed_question_count}')
print(f'Formatting rows merged into generation: {formatting_count}')
print('Tier formula mismatches:', int((df['computed_tier'] != df['tier']).sum()))
print('Max complexity score deviation:', float(np.abs(df['computed_complexity_score'] - df['complexity_score']).max()))
print('Duplicate prompts:', int(df['prompt'].duplicated().sum()))
print('\nTier counts:')
print(df['tier'].value_counts().sort_index())
print('\nIntent counts:')
print(df['intent'].value_counts())
print('\nTask type counts:')
print(df['task_type'].value_counts())

Question task_type fixes applied: 0
Formatting rows merged into generation: 2
Tier formula mismatches: 0
Max complexity score deviation: 1.1102230246251565e-16
Duplicate prompts: 0

Tier counts:
tier
T1     942
T2    1018
T3     313
Name: count, dtype: int64

Intent counts:
intent
ANALYTICAL    1306
FACTUAL        413
SYNTHETIC      380
STRATEGIC      174
Name: count, dtype: int64

Task type counts:
task_type
reasoning         1226
generation         768
summarisation      112
classification      90
coding              77
Name: count, dtype: int64


## 42 Hand-Crafted Features

These combine the missing Phase 1 v4 feature families with the Phase 2-specific features for intent, task type, and reasoning-chain detection.

In [11]:
ARTIFACT_TERMS = ['csv', 'json', 'pdf', 'log', 'yaml', 'yml', 'xlsx', 'docx', 'transcript', 'diagram']
CLOUD_PROVIDERS = ['aws', 'azure', 'gcp', 'google cloud', 'oci']
SYSTEMS = ['salesforce', 'servicenow', 'jira', 'workday', 'sap', 'snowflake', 'databricks', 'okta', 'hubspot', 'github', 'gitlab']
FRAMEWORKS = ['itil', 'finops', 'togaf', 'owasp', 'dora', 'nist', 'hipaa', 'soc 2', 'soc2', 'gdpr', 'iso 27001', 'pci-dss', 'pci dss']
VENDOR_TOOLS = sorted(set(CLOUD_PROVIDERS + SYSTEMS + ['openai', 'anthropic', 'bedrock', 'terraform', 'kubernetes', 'docker', 'jenkins', 'splunk']))

DOMAIN_BUCKETS = {
    'cloud': ['aws', 'azure', 'gcp', 'cloud', 'kubernetes', 'terraform'],
    'finops': ['finops', 'cost', 'budget', 'chargeback', 'showback'],
    'security': ['security', 'vulnerability', 'iam', 'zero trust', 'soc'],
    'devops': ['devops', 'ci/cd', 'pipeline', 'sre', 'deployment'],
    'data': ['data pipeline', 'etl', 'warehouse', 'lakehouse', 'spark'],
    'ai': ['ai', 'llm', 'genai', 'machine learning', 'model'],
    'hr': ['hr', 'employee', 'talent', 'workforce', 'recruiting'],
    'supply': ['supply chain', 'inventory', 'procurement', 'logistics'],
}


def count_regex(text, patterns):
    return sum(len(re.findall(pattern, text)) for pattern in patterns)


def count_terms(text, terms):
    return sum(1 for term in terms if term in text)


def get_style_at(phrasing_styles, i):
    if phrasing_styles is None:
        return None
    try:
        value = phrasing_styles.iloc[i]
    except AttributeError:
        value = phrasing_styles[i]
    return None if pd.isna(value) else str(value).strip().lower()


def handcrafted_features(prompts, phrasing_styles=None):
    rows = []
    for i, prompt in enumerate(prompts):
        text = str(prompt)
        lower = text.lower()
        words = re.findall(r'\b\w+\b', lower)
        unique_words = set(words)
        sentences = [s for s in re.split(r'[.!?]+', text) if s.strip()]
        lines = [line for line in text.splitlines() if line.strip()]
        style = get_style_at(phrasing_styles, i)

        row = {}

        # Text statistics: 6
        row['char_len'] = len(text)
        row['word_count'] = len(words)
        row['sentence_count'] = max(1, len(sentences))
        row['avg_word_len'] = float(np.mean([len(w) for w in words])) if words else 0.0
        row['unique_word_ratio'] = len(unique_words) / max(1, len(words))
        row['line_count'] = len(lines)

        # D5 Context Requirement: 4
        row['has_attachment'] = int(any(term in lower for term in ['uploaded', 'attached', 'provided file', 'document below', 'context below', 'see below']))
        row['provided_artifact_count'] = count_terms(lower, ARTIFACT_TERMS)
        row['large_context_signal'] = int(any(term in lower for term in ['across all', 'entire', 'all of our', 'company-wide', 'large context', 'full document']))
        row['multi_document_signal'] = int(any(term in lower for term in ['multiple', 'all the', 'each of the', 'various', 'several documents', 'set of files']))

        # D3 Output Formality: 4
        row['has_formal_deliverable'] = int(any(term in lower for term in ['report', 'brief', 'proposal', 'specification', 'whitepaper', 'requirements doc']))
        row['has_report_package'] = int(any(term in lower for term in ['appendix', 'table of contents', 'risk register', 'executive summary', 'roadmap', 'implementation plan']))
        row['has_long_output_signal'] = int(any(term in lower for term in ['comprehensive', 'detailed', 'thorough', 'in-depth', 'end-to-end']))
        row['structured_section_count'] = count_terms(lower, ['executive summary', 'timeline', 'roadmap', 'risk register', 'assumptions', 'recommendations', 'next steps', 'success metrics'])

        # D1 Semantic Complexity: 3
        row['has_scope_words'] = int(any(term in lower for term in ['strategic', 'cross-domain', 'enterprise-wide', 'synthesize', 'multi-cloud', 'governance']))
        row['action_verb_count'] = count_terms(lower, ['build', 'design', 'evaluate', 'integrate', 'optimize', 'develop', 'assess', 'recommend', 'compare'])
        row['multi_stage_signal'] = int(bool(re.search(r'\bphase\b|\bstage\b|\bstep\s*1\b|\bmilestone\b|\bsequentially\b|\bfirst\b.*\bthen\b', lower)))

        # D2 Domain Specificity: 4
        row['has_compliance'] = int(any(term in lower for term in ['nist', 'hipaa', 'soc2', 'soc 2', 'gdpr', 'iso 27001', 'pci-dss', 'pci dss', 'compliance']))
        row['cloud_providers_mentioned'] = count_terms(lower, CLOUD_PROVIDERS)
        row['systems_mentioned'] = count_terms(lower, SYSTEMS)
        row['domain_framework_count'] = count_terms(lower, FRAMEWORKS)

        # D4 Research Dependency: 5
        row['external_data_score'] = count_terms(lower, ['market research', 'industry report', 'analyst', 'third-party', 'external data', 'latest', 'current'])
        row['has_time_reference'] = int(bool(re.search(r'\b20\d{2}\b|\bfy\d{2}\b|\bthis quarter\b|\blatest\b|\bcurrent\b|\brecent\b|\btoday\b|\bnow\b', lower)))
        row['vendor_tool_count'] = count_terms(lower, VENDOR_TOOLS)
        row['has_market_terms'] = int(any(term in lower for term in ['competitor', 'market share', 'tam', 'sam', 'som', 'benchmark', 'industry trend']))
        row['has_cost_comparison'] = int(any(term in lower for term in ['pricing', 'cost analysis', 'tco', 'roi', 'showback', 'chargeback', 'cheapest']))

        # Boundary/Risk: 3
        row['has_comparison'] = int(any(term in lower for term in ['compare', 'versus', ' vs ', 'difference between', 'tradeoff']))
        row['stakeholder_mentions'] = count_terms(lower, ['ceo', 'cto', 'cio', 'cfo', 'board', 'leadership', 'management', 'executive'])
        row['risk_language'] = count_terms(lower, ['risk', 'threat', 'vulnerability', 'mitigation', 'breach', 'exposure', 'audit'])

        # Phase 2 Intent / Reasoning Chain: 5
        row['has_role_prompt'] = int(bool(re.search(r'\byou are\b|\bact as\b|\bassume the role\b', lower)))
        row['has_step_request'] = int(bool(re.search(r'\bstep[- ]by[- ]step\b|\bfirst\b.*\bthen\b|\bsequentially\b', lower)))
        row['has_chain_of_thought'] = int(bool(re.search(r'\bthink through\b|\breason about\b|\blet.s think\b|\bchain of thought\b|\bwalk me through\b', lower)))
        if '?' not in text:
            row['question_complexity'] = 0
        elif any(term in lower for term in ['what should', 'recommend', 'design a', 'propose', 'strategy']):
            row['question_complexity'] = 3
        elif any(term in lower for term in ['why', 'how', 'compare', 'analyze', 'evaluate', 'assess']):
            row['question_complexity'] = 2
        else:
            row['question_complexity'] = 1
        row['multi_domain_count'] = sum(1 for bucket_terms in DOMAIN_BUCKETS.values() if any(term in lower for term in bucket_terms))

        # Phase 2 Task Type: 5
        row['has_code_block'] = int('```' in text)
        row['has_output_format'] = int(bool(re.search(r'\bin json\b|\bas a table\b|\bformat as\b|\bcsv output\b|\bin yaml\b|\bas markdown\b', lower)))
        row['has_creative_language'] = int(any(term in lower for term in ['imagine', 'creative', 'story', 'write a', 'compose', 'fictional']))
        row['has_classification_request'] = int(any(term in lower for term in ['classify', 'categorize', 'label', 'which category', 'sort into']))
        row['enumeration_signal'] = int(bool(re.search(r'\blist\b|\btop \d+\b|\benumerate\b|\bbullet point\b|\brank\b', lower)))

        # Phrasing style: 3
        row['phrasing_explicit'] = int(style == 'explicit')
        row['phrasing_implicit'] = int(style == 'implicit')
        row['phrasing_vague'] = int(style == 'vague')

        rows.append(row)

    feature_df = pd.DataFrame(rows).fillna(0)
    expected_features = 42
    if feature_df.shape[1] != expected_features:
        raise ValueError(f'Expected {expected_features} hand-crafted features, got {feature_df.shape[1]}')
    return feature_df

hand_df = handcrafted_features(df['prompt'], df.get('phrasing_style'))
print('Hand-crafted feature shape:', hand_df.shape)
hand_df.head()

Hand-crafted feature shape: (2273, 42)


,char_len,word_count,sentence_count,avg_word_len,unique_word_ratio,line_count,has_attachment,provided_artifact_count,large_context_signal,multi_document_signal,...,question_complexity,multi_domain_count,has_code_block,has_output_format,has_creative_language,has_classification_request,enumeration_signal,phrasing_explicit,phrasing_implicit,phrasing_vague
0,1058,158,11,5.506329,0.696203,4,0,1,0,0,...,0,3,0,0,1,0,0,0,0,0
1,531,82,9,5.146341,0.707317,1,0,1,0,0,...,0,1,0,0,0,0,1,0,0,0
2,525,70,5,6.371429,0.814286,1,0,1,0,0,...,0,1,0,0,0,0,0,0,0,0
3,237,38,4,5.078947,0.921053,1,0,0,0,0,...,0,1,0,0,0,0,1,0,0,0
4,713,107,12,5.364486,0.644860,1,0,1,0,0,...,0,2,0,0,0,0,0,0,0,0


## Embeddings

Embeddings are computed once. During CV, PCA and scaling are fit inside each fold to avoid leaking validation distribution information into feature preprocessing.

In [12]:
embedding_model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

prompts = df['prompt'].astype(str).tolist()
embeddings = embedding_model.encode(
    prompts,
    batch_size=64,
    show_progress_bar=True,
    normalize_embeddings=True,
)

print('Embedding shape:', embeddings.shape)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/36 [00:00<?, ?it/s]

Embedding shape: (2273, 384)


## Feature Matrix Helpers

In [13]:
def fit_shared_transformers(train_embeddings, train_hand_features):
    pca = PCA(n_components=PCA_COMPONENTS, random_state=RANDOM_STATE)
    train_emb_pca = pca.fit_transform(train_embeddings)
    train_raw = np.hstack([train_emb_pca, train_hand_features])

    scaler = StandardScaler()
    train_X = scaler.fit_transform(train_raw)
    return pca, scaler, train_X


def transform_shared_features(embeddings_part, hand_features_part, pca, scaler):
    emb_pca = pca.transform(embeddings_part)
    raw = np.hstack([emb_pca, hand_features_part])
    return scaler.transform(raw)


def fit_full_feature_matrix():
    pca, scaler, X_full = fit_shared_transformers(embeddings, hand_df.values)
    print('PCA variance explained:', round(float(pca.explained_variance_ratio_.sum()), 4))
    print('Base feature shape:', X_full.shape)
    return pca, scaler, X_full

## Label Encoding

In [14]:
label_encoders = {}
targets = {}

for col in SCORE_COLS:
    unknown_scores = sorted(set(df[col].dropna().astype(float)) - set(VALID_SCORES))
    if unknown_scores:
        raise ValueError(f'{col} has invalid scores: {unknown_scores}')
    targets[col] = df[col].astype(float).map(SCORE_TO_CLASS).astype(int).values

for col in ['tier', 'intent', 'task_type']:
    le = LabelEncoder()
    targets[col] = le.fit_transform(df[col].astype(str))
    label_encoders[col] = le
    print(f'{col}: {list(le.classes_)}')

targets['reasoning_chain_detected'] = df['reasoning_chain_detected'].astype(bool).astype(int).values

head_classes = {
    'tier': len(label_encoders['tier'].classes_),
    'intent': len(label_encoders['intent'].classes_),
    'task_type': len(label_encoders['task_type'].classes_),
    'reasoning_chain_detected': 2,
    'd1': 5,
    'd2': 5,
    'd3': 5,
    'd4': 5,
    'd5': 5,
}

STAGE1_HEADS = ['tier', 'intent', 'task_type', 'reasoning_chain_detected']
ALL_HEADS = STAGE1_HEADS + SCORE_COLS
print('Head classes:', head_classes)

tier: ['T1', 'T2', 'T3']
intent: ['ANALYTICAL', 'FACTUAL', 'STRATEGIC', 'SYNTHETIC']
task_type: ['classification', 'coding', 'generation', 'reasoning', 'summarisation']
Head classes: {'tier': 3, 'intent': 4, 'task_type': 5, 'reasoning_chain_detected': 2, 'd1': 5, 'd2': 5, 'd3': 5, 'd4': 5, 'd5': 5}


## Model Helpers

In [15]:
def make_xgb(num_classes, seed=RANDOM_STATE):
    is_binary = num_classes == 2
    params = dict(
        n_estimators=300,
        max_depth=3,
        learning_rate=0.05,
        subsample=0.85,
        colsample_bytree=0.85,
        reg_lambda=2.0,
        min_child_weight=5,
        random_state=seed,
        eval_metric='logloss' if is_binary else 'mlogloss',
        objective='binary:logistic' if is_binary else 'multi:softprob',
        tree_method='hist',
    )
    if not is_binary:
        params['num_class'] = num_classes
    return XGBClassifier(**params)


def fit_head(name, X_train, y_train, seed=RANDOM_STATE):
    model = make_xgb(head_classes[name], seed=seed)
    sample_weight = compute_sample_weight(class_weight='balanced', y=y_train)
    model.fit(X_train, y_train, sample_weight=sample_weight)
    return model


def train_two_stage_heads(X_train, train_idx, seed=RANDOM_STATE):
    heads = {}

    for name in STAGE1_HEADS:
        y_train = targets[name][train_idx]
        heads[name] = fit_head(name, X_train, y_train, seed=seed)

    tier_pred_train = heads['tier'].predict(X_train).reshape(-1, 1)
    X_train_aug = np.hstack([X_train, tier_pred_train])

    for name in SCORE_COLS:
        y_train = targets[name][train_idx]
        heads[name] = fit_head(name, X_train_aug, y_train, seed=seed)

    return heads


def predict_head(heads, name, X_base):
    if name in SCORE_COLS:
        tier_pred = heads['tier'].predict(X_base).reshape(-1, 1)
        X_aug = np.hstack([X_base, tier_pred])
        return heads[name].predict(X_aug)
    return heads[name].predict(X_base)


def predict_all_dims(heads, X_base):
    tier_pred = heads['tier'].predict(X_base).reshape(-1, 1)
    X_aug = np.hstack([X_base, tier_pred])
    pred_dim_classes = {col: heads[col].predict(X_aug) for col in SCORE_COLS}
    return pd.DataFrame({col: [CLASS_TO_SCORE[int(v)] for v in pred_dim_classes[col]] for col in SCORE_COLS})

## 5-Fold Stratified Cross-Validation

Splits are stratified by `tier`, the main routing target.

In [16]:
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
fold_results = {name: {'accuracy': [], 'macro_f1': []} for name in ALL_HEADS}
fold_results['formula_tier'] = {'accuracy': [], 'macro_f1': []}

tier_confusions = []
formula_tier_confusions = []

for fold, (train_idx, val_idx) in enumerate(skf.split(df, targets['tier']), start=1):
    print(f'\n=== Fold {fold}/{N_SPLITS} ===')

    train_embeddings = embeddings[train_idx]
    val_embeddings = embeddings[val_idx]
    train_hand = hand_df.iloc[train_idx].values
    val_hand = hand_df.iloc[val_idx].values

    fold_pca, fold_scaler, X_train = fit_shared_transformers(train_embeddings, train_hand)
    X_val = transform_shared_features(val_embeddings, val_hand, fold_pca, fold_scaler)

    heads = train_two_stage_heads(X_train, train_idx, seed=RANDOM_STATE + fold)

    for name in ALL_HEADS:
        y_true = targets[name][val_idx]
        y_pred = predict_head(heads, name, X_val)
        fold_results[name]['accuracy'].append(accuracy_score(y_true, y_pred))
        fold_results[name]['macro_f1'].append(f1_score(y_true, y_pred, average='macro', zero_division=0))

    direct_tier_pred = predict_head(heads, 'tier', X_val)
    tier_confusions.append(confusion_matrix(targets['tier'][val_idx], direct_tier_pred, labels=np.arange(head_classes['tier'])))

    pred_dims = predict_all_dims(heads, X_val)
    pred_dims['complexity_score'] = complexity_score_from_dims_frame(pred_dims)
    formula_tier_labels = pred_dims['complexity_score'].apply(tier_from_score).values
    formula_tier_pred = label_encoders['tier'].transform(formula_tier_labels)
    y_tier_true = targets['tier'][val_idx]
    fold_results['formula_tier']['accuracy'].append(accuracy_score(y_tier_true, formula_tier_pred))
    fold_results['formula_tier']['macro_f1'].append(f1_score(y_tier_true, formula_tier_pred, average='macro', zero_division=0))
    formula_tier_confusions.append(confusion_matrix(y_tier_true, formula_tier_pred, labels=np.arange(head_classes['tier'])))

    print('Tier acc:', round(fold_results['tier']['accuracy'][-1], 4))
    print('Formula tier acc:', round(fold_results['formula_tier']['accuracy'][-1], 4))

print('\n' + '=' * 72)
print('5-Fold CV Results (mean +/- std)')
print('=' * 72)
for name in ALL_HEADS + ['formula_tier']:
    acc = np.array(fold_results[name]['accuracy'])
    f1 = np.array(fold_results[name]['macro_f1'])
    print(f'{name:30s} Acc: {acc.mean():.4f} +/- {acc.std():.4f}   F1: {f1.mean():.4f} +/- {f1.std():.4f}')

print('\nAggregate direct tier confusion:')
tier_labels = label_encoders['tier'].classes_
print(pd.DataFrame(np.sum(tier_confusions, axis=0), index=tier_labels, columns=tier_labels))

print('\nAggregate formula-derived tier confusion:')
print(pd.DataFrame(np.sum(formula_tier_confusions, axis=0), index=tier_labels, columns=tier_labels))


=== Fold 1/5 ===
Tier acc: 0.8044
Formula tier acc: 0.8066

=== Fold 2/5 ===
Tier acc: 0.811
Formula tier acc: 0.8022

=== Fold 3/5 ===
Tier acc: 0.7824
Formula tier acc: 0.7846

=== Fold 4/5 ===
Tier acc: 0.793
Formula tier acc: 0.7885

=== Fold 5/5 ===
Tier acc: 0.7907
Formula tier acc: 0.7885

5-Fold CV Results (mean +/- std)
tier                           Acc: 0.7963 +/- 0.0102   F1: 0.7964 +/- 0.0093
intent                         Acc: 0.8306 +/- 0.0098   F1: 0.7939 +/- 0.0133
task_type                      Acc: 0.7215 +/- 0.0050   F1: 0.6124 +/- 0.0483
reasoning_chain_detected       Acc: 0.8988 +/- 0.0065   F1: 0.8538 +/- 0.0117
d1                             Acc: 0.6568 +/- 0.0128   F1: 0.6848 +/- 0.0193
d2                             Acc: 0.7176 +/- 0.0112   F1: 0.6482 +/- 0.0105
d3                             Acc: 0.7528 +/- 0.0222   F1: 0.6996 +/- 0.0303
d4                             Acc: 0.7897 +/- 0.0112   F1: 0.6788 +/- 0.0200
d5                             Acc: 0.7422 +

## Final Full-Dataset Training

After CV, train the final v2 model on all rows so the inference function can be used directly.

In [17]:
final_pca, final_scaler, X_full = fit_full_feature_matrix()
final_train_idx = np.arange(len(df))
heads = train_two_stage_heads(X_full, final_train_idx, seed=RANDOM_STATE)

print('Final models trained:', list(heads.keys()))

PCA variance explained: 0.5191
Base feature shape: (2273, 82)
Final models trained: ['tier', 'intent', 'task_type', 'reasoning_chain_detected', 'd1', 'd2', 'd3', 'd4', 'd5']


## Rule-Based Research Signals

In [18]:
RESEARCH_SIGNAL_KEYWORDS = {
    'market_research': ['market', 'industry', 'trend', 'tam', 'sam', 'som'],
    'competitive_analysis': ['competitor', 'competitive', 'benchmark', 'rival'],
    'regulatory_compliance': ['regulation', 'regulatory', 'compliance', 'gdpr', 'hipaa', 'sox', 'eu ai act'],
    'security': ['security', 'vulnerability', 'threat', 'risk', 'iam', 'zero trust'],
    'cloud_infrastructure': ['aws', 'azure', 'gcp', 'cloud', 'kubernetes', 'terraform'],
    'finops': ['finops', 'cost', 'spend', 'budget', 'showback', 'chargeback'],
    'devops': ['ci/cd', 'pipeline', 'deployment', 'sre', 'devops', 'observability'],
    'data_engineering': ['data pipeline', 'etl', 'warehouse', 'lakehouse', 'spark'],
    'ai_governance': ['ai governance', 'llm', 'model risk', 'genai', 'guardrail'],
    'system_integration': ['integration', 'api', 'webhook', 'middleware'],
    'supply_chain': ['supply chain', 'inventory', 'procurement', 'logistics'],
    'hr_tech': ['hr', 'employee', 'workforce', 'talent', 'recruiting'],
    'vendor_analysis': ['vendor', 'rfi', 'rfp', 'procurement'],
}


def extract_research_signals(prompt, d4_score):
    if d4_score <= 0:
        return []
    text = str(prompt).lower()
    signals = []
    for signal, keywords in RESEARCH_SIGNAL_KEYWORDS.items():
        if any(keyword in text for keyword in keywords):
            signals.append(signal)
    return signals if signals else ['external_research']

## Inference Function

In [19]:
def build_features_for_prompts(new_prompts):
    new_prompts = [str(prompt) for prompt in new_prompts]
    new_embeddings = embedding_model.encode(
        new_prompts,
        batch_size=64,
        show_progress_bar=False,
        normalize_embeddings=True,
    )
    new_hand = handcrafted_features(new_prompts, phrasing_styles=None)
    return transform_shared_features(new_embeddings, new_hand.values, final_pca, final_scaler)


def max_probability(model, X_part):
    proba = model.predict_proba(X_part)[0]
    return float(np.max(proba))


def predict_prompt(prompt):
    X_one = build_features_for_prompts([prompt])

    tier_class = int(heads['tier'].predict(X_one)[0])
    direct_tier = label_encoders['tier'].inverse_transform([tier_class])[0]
    intent = label_encoders['intent'].inverse_transform(heads['intent'].predict(X_one))[0]
    task_type = label_encoders['task_type'].inverse_transform(heads['task_type'].predict(X_one))[0]
    reasoning_chain = bool(int(heads['reasoning_chain_detected'].predict(X_one)[0]))

    X_one_aug = np.hstack([X_one, np.array([[tier_class]])])
    predicted_dims = {}
    for col in SCORE_COLS:
        pred_class = int(heads[col].predict(X_one_aug)[0])
        predicted_dims[col] = CLASS_TO_SCORE[pred_class]

    score = (
        predicted_dims['d1'] * 0.35 +
        predicted_dims['d2'] * 0.20 +
        predicted_dims['d3'] * 0.20 +
        predicted_dims['d4'] * 0.15 +
        predicted_dims['d5'] * 0.10
    )

    key_confidences = [
        max_probability(heads['tier'], X_one),
        max_probability(heads['intent'], X_one),
        max_probability(heads['task_type'], X_one),
        max_probability(heads['reasoning_chain_detected'], X_one),
    ]
    confidence = float(np.mean(key_confidences))

    result = {}
    for col in SCORE_COLS:
        result[col] = predicted_dims[col]
        result[f'{col}_label'] = DIMENSION_LABELS[col]

    result.update({
        'complexity_score': round(float(score), 4),
        'tier': direct_tier,
        'formula_tier': tier_from_score(score),
        'intent': intent,
        'task_type': task_type,
        'reasoning_chain_detected': reasoning_chain,
        'research_signals': extract_research_signals(prompt, predicted_dims['d4']),
        'confidence': round(confidence, 4),
    })
    return result

In [20]:
sample_prompt = 'Design a multi-cloud GenAI governance architecture for a Fortune 500 company, including compliance risks and vendor evaluation criteria.'
print(json.dumps(predict_prompt(sample_prompt), indent=2))

{
  "d1": 1.0,
  "d1_label": "Semantic Complexity",
  "d2": 1.0,
  "d2_label": "Domain Specificity",
  "d3": 1.0,
  "d3_label": "Output Formality",
  "d4": 0.75,
  "d4_label": "Research Dependency",
  "d5": 0.75,
  "d5_label": "Context Requirement",
  "complexity_score": 0.9375,
  "tier": "T3",
  "formula_tier": "T3",
  "intent": "STRATEGIC",
  "task_type": "reasoning",
  "reasoning_chain_detected": true,
  "research_signals": [
    "regulatory_compliance",
    "security",
    "cloud_infrastructure",
    "ai_governance",
    "vendor_analysis"
  ],
  "confidence": 0.9072
}


## Optional Detailed Reports on a Final Holdout

The main benchmark above is 5-fold CV. Use this section only if you want an additional quick sanity check after full-model training changes.

## Notes for v3

- Add real `formatting` and `sparql_generation` samples if those task types matter.
- Consider replacing rule-based `research_signals` with a multi-label model after signal audit.
- Compare XGBoost against LightGBM/CatBoost if available.
- Calibrate confidence with isotonic or Platt scaling if production confidence thresholds are needed.